# Cyclomatic Complexity (CC)

In [ ]:
import json


In [14]:
# Read and process data from 'CC.txt' into a structured list

def load_cc_data(file_path: str):
    """
    Load and clean data from a text file.

    Each line is stripped of whitespace and '|' characters.
    Values are split into a list, the first element is cleaned
    (removing quotes, replacing '.' with '/'), and the rest are
    converted to floats.
    
    Returns:
        List of lists: [[str, float, float, ...], ...]
    """
    with open(file_path, 'r') as f:
        lines = f.readlines()

    cleaned_data = []
    for line in lines:
        # Skip empty lines
        line = line.strip()
        if not line:
            continue

        # Remove '|' characters
        line = line.replace('|', '')

        # Split line into individual values
        values = line.split()

        # Process first value (string) and the rest (floats)
        first_val = values[0].strip("'").replace('.', '/')
        numeric_vals = [float(v.strip("'")) for v in values[1:]]
        cleaned_data.append([first_val] + numeric_vals)

    return cleaned_data

# Load the data
data_cc = load_cc_data('CC.txt')


In [3]:
with open('CC.txt', 'r') as f:
    data = f.readlines()
    # Bỏ dòng trống, bỏ ký tự |, strip khoảng trắng
    data_cc = [line.strip().replace('|', '') for line in data if line.strip()]
    # Tách từng dòng thành list các giá trị
    data_cc = [line.split() for line in data_cc]
    # Nếu muốn xử lý tiếp (ví dụ: bỏ dấu nháy đơn ở từng giá trị)
    data_cc = [[x[0].strip("'").replace('.', '/')] + [float(item.strip("'")) for item in x[1:]] for x in data_cc]

In [18]:
import json

# --- Grouping data_cc based on CC ranges ---
gr1 = [x for x in data_cc if x[1] <= 50]
gr2 = [x for x in data_cc if 50 < x[1] <= 100]
gr3 = [x for x in data_cc if 100 < x[1] <= 200]
gr4 = [x for x in data_cc if x[1] > 200]

# --- Function to check coverage quality for a group ---
def check_quality(file_path: str, group: list) -> str:
    """
    Compute code coverage score for a specific group of files.

    Args:
        file_path (str): Path to the file containing JSON results per line.
        group (list): A list of file information, where the first element is the file identifier.

    Returns:
        str: Formatted coverage score as a percentage.
    """
    covered_lines = 0
    total_lines = 0

    print(f"Number of files in this group: {len(group)}")

    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                # Extract JSON part from the line
                brace_pos = line.find('{')
                if brace_pos == -1:
                    print(f"Warning: No JSON found in line: {line}")
                    continue

                data = json.loads(line[brace_pos:])
                file_name = data.get('file', '')

                # Check if any file in group matches the current file
                for file_info in group:
                    if file_info[0] in file_name:
                        covered_lines += data.get('covered_lines', 0)
                        total_lines += data.get('total_lines', 0)

            except json.JSONDecodeError:
                print(f"JSON decode error in line: {line}")
            except Exception as e:
                print(f"Unexpected error: {e} | Line: {line}")

    print(f"Total covered lines in group: {covered_lines}")
    print(f"Total lines in group: {total_lines}")

    coverage_score = (covered_lines / total_lines * 100) if total_lines else 0
    return f"Coverage score of this group: {coverage_score:.2f}%"

# --- Example usage ---
print(check_quality('testweaver_result.txt', gr4))


Number of files in this group: 13
Total covered lines in group: 5371
Total lines in group: 8121.0
Coverage score of this group: 66.14%


# File Size

In [19]:
import json

def cal_file_size(file_path: str, start: int, end: int) -> float:
    """
    Calculate coverage score for files within a specific size range.

    Args:
        file_path (str): Path to the file containing JSON results per line.
        start (int): Minimum number of lines to include.
        end (int): Maximum number of lines to include.

    Returns:
        float: Coverage ratio (covered lines / total lines) for files in the range.
    """
    num_files = 0
    covered_lines = 0
    total_lines = 0

    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                # Extract JSON part from the line
                brace_pos = line.find('{')
                if brace_pos == -1:
                    print(f"Warning: No JSON found in line: {line}")
                    continue

                data = json.loads(line[brace_pos:])
                num_line = data.get('total_lines', 0)

                # Check if file size falls within the specified range
                if start <= num_line <= end:
                    num_files += 1
                    covered_lines += data.get('covered_lines', 0)
                    total_lines += num_line

            except json.JSONDecodeError:
                print(f"JSON decode error in line: {line}")
            except Exception as e:
                print(f"Unexpected error: {e} | Line: {line}")

    coverage_score = (covered_lines / total_lines) * 100 if total_lines else 0
    print(f"Number of files with size from {start} to {end}: {num_files}")
    print(f"Coverage score for these files: {coverage_score:.2f}%")

    return covered_lines / total_lines if total_lines else 0


In [22]:
cal_file_size('testweaver_result.txt', 150, 500)

Number of files with size from 150 to 500: 109
Coverage score for these files: 67.15%


0.671544061162527